# 🔮 Gemma 4 on Colab — Ollama CLI

Run **Google Gemma 4** locally on Colab using **Ollama**.

### Before you start
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Run all cells below in order

### Model selection (auto-detected)
| GPU VRAM | Model pulled | Size |
|----------|-------------|------|
| ≥ 16 GB  | `gemma4:e4b` | 9.6 GB |
| < 16 GB  | `gemma4:e2b` | 7.2 GB |

---

## 1️⃣ Install Ollama

In [ ]:
%%bash
# ── Install Ollama ───────────────────────────────────────────────────────
curl -fsSL https://ollama.com/install.sh | sh
echo '✅ Ollama installed'

## 2️⃣ Start Ollama server (background)

In [ ]:
import subprocess, time, requests

# Start Ollama in the background
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=open("/tmp/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)
print(f"Ollama server starting (PID: {proc.pid})...")

# Wait for it to be ready
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("✅ Ollama server is ready!")
            break
    except:
        pass
    time.sleep(1)
else:
    print("❌ Ollama failed to start. Check /tmp/ollama.log")

## 3️⃣ Detect GPU & pull Gemma 4

In [ ]:
import subprocess, re

# Detect GPU VRAM
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
        text=True
    ).strip()
    gpu_name, vram_mb = out.split(", ")
    vram_mb = int(vram_mb)
    print(f"🖥️  GPU: {gpu_name}")
    print(f"💾 VRAM: {vram_mb} MB ({vram_mb / 1024:.1f} GB)")
except Exception as e:
    print(f"⚠️ Could not detect GPU: {e}")
    vram_mb = 0

# Pick the best model that fits
if vram_mb >= 16000:
    MODEL = "gemma4:e4b"   # 9.6 GB — fits comfortably on 16GB+
else:
    MODEL = "gemma4:e2b"   # 7.2 GB — fits on T4 (15 GB)

print(f"\n📦 Pulling {MODEL} ...")
print("(This may take 3-5 minutes on the first run)\n")

!ollama pull {MODEL}

print(f"\n✅ {MODEL} is ready!")

## 4️⃣ Quick test — make sure it works

In [ ]:
import requests, json

# Quick non-streaming test
r = requests.post("http://localhost:11434/api/chat", json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Say 'Hello! Gemma 4 is working!' and nothing else."}],
    "stream": False
}, timeout=120)

reply = r.json()["message"]["content"]
print(f"🤖 Gemma 4 says: {reply}")

## 5️⃣ Interactive CLI Chat

Run the cell below to start chatting with Gemma 4.

- Type your message and press **Enter**
- Type **`quit`** or **`exit`** to stop
- Type **`clear`** to reset conversation history
- Responses stream token-by-token

---

In [ ]:
import requests, json, sys
from IPython.display import clear_output

OLLAMA_URL = "http://localhost:11434/api/chat"

SYSTEM_PROMPT = (
    "You are Gemma 4, a highly capable AI assistant built by Google DeepMind. "
    "You are helpful, harmless, and honest. Use markdown formatting when helpful. "
    "Be concise but thorough."
)

history = []

print("═" * 60)
print("  🔮 Gemma 4 CLI Chat")
print(f"  Model: {MODEL}")
print("  Commands: quit, exit, clear")
print("═" * 60)
print()

while True:
    try:
        user_input = input("You: ").strip()
    except (EOFError, KeyboardInterrupt):
        print("\n👋 Bye!")
        break

    if not user_input:
        continue

    if user_input.lower() in ("quit", "exit"):
        print("👋 Bye!")
        break

    if user_input.lower() == "clear":
        history = []
        print("🗑️  History cleared.\n")
        continue

    # Add user message to history
    history.append({"role": "user", "content": user_input})

    # Build messages with system prompt
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history

    # Stream the response
    print("\nGemma 4: ", end="", flush=True)
    full_response = ""

    try:
        with requests.post(
            OLLAMA_URL,
            json={"model": MODEL, "messages": messages, "stream": True},
            stream=True,
            timeout=300,
        ) as r:
            r.raise_for_status()
            for line in r.iter_lines(decode_unicode=True):
                if not line:
                    continue
                chunk = json.loads(line)
                token = chunk.get("message", {}).get("content", "")
                if token:
                    print(token, end="", flush=True)
                    full_response += token

        # Save assistant response to history
        history.append({"role": "assistant", "content": full_response})

        # Keep history manageable (last 20 turns)
        if len(history) > 40:
            history = history[-20:]

        print("\n")

    except requests.ConnectionError:
        print("\n⚠️ Lost connection to Ollama. Is the server still running?")
        print("   Re-run Cell 2 to restart it.\n")
        history.pop()  # Remove the failed user message
    except Exception as e:
        print(f"\n⚠️ Error: {e}\n")
        history.pop()

---

## 🔧 Utilities

In [ ]:
# ── Check which models are installed ─────────────────────────────────────
!ollama list

In [ ]:
# ── Check GPU usage ──────────────────────────────────────────────────────
!nvidia-smi

In [ ]:
# ── Pull a different model (e.g. upgrade to e4b or try 26b on A100) ─────
# !ollama pull gemma4:e4b
# MODEL = "gemma4:e4b"  # then re-run the chat cell

In [ ]:
# ── View Ollama server logs ───────────────────────────────────────────────
!tail -30 /tmp/ollama.log